# Logit averaging ensemble

import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import os
import pandas as pd

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

# Device (auto GPU / CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ResNet50
resnet_model = models.resnet50(weights=None)

# Replace final classifier for 4 classes
resnet_model.fc = nn.Linear(resnet_model.fc.in_features, 4)

# Load trained weights safely for CPU/GPU
state_dict = torch.load("FYP/resnet50_basic.pth", map_location=device)
resnet_model.load_state_dict(state_dict)

# Move model to device
resnet_model = resnet_model.to(device)

# Evaluation mode
resnet_model.eval()

In [ ]:
# EfficientNetB0
efficient_model = models.efficientnet_b0(pretrained=False)

# Freeze all conv layers
for param in efficient_model.features.parameters():
    param.requires_grad = False

# Replace classifier for 4 classes
efficient_model.classifier[1] = nn.Linear(efficient_model.classifier[1].in_features, 4)

# Load trained weights
efficient_model.load_state_dict(torch.load("FYP/efficientnetB0_basic.pth", map_location=device))

# Move to device and set eval
efficient_model.to(device)
efficient_model.eval()

In [ ]:
from torchvision import transforms

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),   # match your model input size
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
import os

test_root = "/content/FYP/Split/test"  
class_names = ["512Glioma", "512Meningioma", "512Pituitary", "512Normal"]

# Make a list of all test images
test_images = []
for cls in class_names:
    cls_path = os.path.join(test_root, cls)
    for img_name in os.listdir(cls_path):
        test_images.append({
            "path": os.path.join(cls_path, img_name),
            "true_class": cls
        })

# check first 5 entries
test_images[:5]

# Testing

In [ ]:
from PIL import Image
import torch.nn.functional as F

# Pick first test image
img_info = test_images[0]
img = Image.open(img_info["path"]).convert("RGB")
img_tensor = preprocess(img).unsqueeze(0).to(device)  # add batch dimension

with torch.no_grad():
    resnet_logits = resnet_model(img_tensor)
    effnet_logits = efficient_model(img_tensor)

    # Logit averaging
    final_logits = (resnet_logits + effnet_logits) / 2
    probs = F.softmax(final_logits, dim=1)
    pred = torch.argmax(probs, dim=1).item()

print("Predicted class:", class_names[pred])
print("True class:", img_info["true_class"])

looping through the test set

In [ ]:
all_preds = []
all_labels = []

for img_info in test_images:
    img = Image.open(img_info["path"]).convert("RGB")
    img_tensor = preprocess(img).unsqueeze(0).to(device)

    with torch.no_grad():
        resnet_logits = resnet_model(img_tensor)
        effnet_logits = efficient_model(img_tensor)

        # Logit averaging
        final_logits = (resnet_logits + effnet_logits) / 2
        probs = F.softmax(final_logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()

    all_preds.append(pred)
    all_labels.append(class_names.index(img_info["true_class"]))

In [ ]:
correct = sum([p == t for p, t in zip(all_preds, all_labels)])
accuracy = correct / len(all_labels)
print("Logit averaging ensemble accuracy:", accuracy)

# Class-wise weighted logit averaging ensemble

In [ ]:
import torch

resnet_weights = torch.tensor([0.5, 0.5, 0.3, 0.7]).to(device)
effnet_weights  = 1 - resnet_weights  # complementary                      

In [ ]:
all_preds_weighted = []
all_labels_weighted = []

for img_info in test_images:
    img = Image.open(img_info["path"]).convert("RGB")
    img_tensor = preprocess(img).unsqueeze(0).to(device)

    with torch.no_grad():
        resnet_logits = resnet_model(img_tensor)
        effnet_logits = efficient_model(img_tensor)

        # Apply class-wise weights
        weighted_logits = resnet_logits * resnet_weights + effnet_logits * effnet_weights

        # Softmax and prediction
        probs = torch.softmax(weighted_logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()

    all_preds_weighted.append(pred)
    all_labels_weighted.append(class_names.index(img_info["true_class"]))

In [ ]:
correct = sum([p == t for p, t in zip(all_preds_weighted, all_labels_weighted)])
accuracy = correct / len(all_labels_weighted)
print("Class-wise weighted logit ensemble accuracy:", accuracy)

## Grid search method to find the best weights 

In [ ]:
import itertools
import torch

# Candidate weights for each class (ResNet)
candidates = [0.6,0.4]

# Generate all combinations for 4 classes
all_combinations = list(itertools.product(candidates, repeat=4))
print(f"Total combinations to try: {len(all_combinations)}")

In [ ]:
import os

val_root = "/content/FYP/Split/val"  # path to validation folder
class_names = ["512Glioma", "512Meningioma", "512Normal", "512Pituitary"]

# Create list of all validation images
val_images = []
for cls in class_names:
    cls_path = os.path.join(val_root, cls)
    for img_name in os.listdir(cls_path):
        val_images.append({
            "path": os.path.join(cls_path, img_name),
            "true_class": cls
        })

# Check first 5 entries
val_images[:5]

In [ ]:
best_acc = 0
best_weights = None

for idx, comb in enumerate(all_combinations, 1):
    print(f"Trying combination {idx}/{len(all_combinations)}: {comb}")
    
    resnet_w = torch.tensor(comb).to(device)  # ResNet weights
    effnet_w = 1 - resnet_w                    # EffNet complementary weights
    correct = 0

    for img_info in val_images:
        img = Image.open(img_info["path"]).convert("RGB")
        img_tensor = preprocess(img).unsqueeze(0).to(device)

        with torch.no_grad():
            r_logits = resnet_model(img_tensor)
            e_logits = efficient_model(img_tensor)
            weighted_logits = r_logits * resnet_w + e_logits * effnet_w
            pred = torch.argmax(torch.softmax(weighted_logits, dim=1), dim=1).item()

        if pred == class_names.index(img_info["true_class"]):
            correct += 1

    acc = correct / len(val_images)
    if acc > best_acc:
        best_acc = acc
        best_weights = comb
        
        # Save intermediate best result
        with open("best_weights_progress.txt", "w") as f:
            f.write(f"{best_acc},{best_weights}\n")

print("Best class-wise weights for ResNet:", best_weights)
print("Corresponding best accuracy:", best_acc)